# Carta Social: geographic export

This workflow joins the service records selected in notebook 02 to equipment
coordinates, then exports a complete CSV and a GeoPackage of valid points.

The pipeline supports all available service types or any explicitly selected
subset. Accessibility matrices are outside the scope of this workflow.


In [ ]:
from pathlib import Path
import sys

# Support execution from the repository root or the notebooks directory.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "scripts" / "utils.py").exists():
    raise FileNotFoundError("Start Jupyter from the ScraperCartaSocial repository.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from scripts.utils import INTERIM, PROCESSED, write_json, ensure_data_directories
from scripts.geography import join_coordinates, make_points, save_geopackage

ensure_data_directories()


## Service records

The export reads `services_selected.csv`, created by notebook 02 from the full
collection or an optional selection. It does not reapply a fixed worksheet
or a predefined list of service types.

Multiple services can share an equipment ID and a location. These records
are retained because the observation unit is the service, not the facility.
English field names are used; source category labels remain unchanged.

Run notebook 02 first to generate both the selected records and their
equipment coordinates.


In [ ]:
selected_services = pd.read_csv(
    INTERIM / "services_selected.csv",
    dtype={"equipment_id": "string"},
)
coordinates = pd.read_csv(
    INTERIM / "equipment_coordinates.csv",
    dtype={"equipment_id": "string"},
)

print("Service records:", len(selected_services))
print("Service types:", selected_services["service_type"].nunique())
print("Distinct equipment:", selected_services["equipment_id"].nunique())


## Coordinate join and coverage

A left join preserves every selected service record. The join validates that
each equipment ID has at most one coordinate record, preventing accidental
multiplication of observations.

The CSV includes unresolved records. Only valid latitude and longitude pairs
produce point geometries, so the spatial export can contain fewer records
than the complete service table.


In [ ]:
service_locations = join_coordinates(selected_services, coordinates)
service_points = make_points(service_locations)

print("Service records before joining:", len(selected_services))
print("Service records after joining:", len(service_locations))
print("Service records with valid points:", len(service_points))
print("Service records without valid points:", len(service_locations) - len(service_points))


## Export names and metadata

`carta_social_services.csv` contains the complete selected service table,
including unresolved coordinates. `carta_social_service_locations.gpkg`
contains only valid point features in the `service_locations` layer.

The names describe the source, observation unit, and geographic content
without assuming a fixed set of categories. A companion metadata JSON
records category counts and spatial coverage for the actual selection.

Coordinates use `EPSG:4326`, with longitude as x and latitude as y, in degrees.
Re-execution replaces the current named outputs. Historical exports are
preserved separately in `archive/processed/`.


In [ ]:
service_locations.to_csv(
    PROCESSED / "carta_social_services.csv",
    index=False,
)

if len(service_points) > 0:
    output_path = save_geopackage(
        service_points,
        PROCESSED / "carta_social_service_locations.gpkg",
        layer="service_locations",
        metadata={
            "title": "Carta Social service locations",
            "description": "One point per selected social-service record with reusable coordinates.",
            "source": "Carta Social",
            "coordinate_quality": "Available coordinates; positional accuracy is not certified.",
        },
    )
    print("GeoPackage saved:", output_path)
else:
    previous_export = PROCESSED / "carta_social_service_locations.gpkg"
    previous_export.unlink(missing_ok=True)
    print("No valid points to export. Review the coordinate results in notebook 02.")

category_counts = selected_services["service_type"].value_counts()
dataset_metadata = {
    "source": "Carta Social",
    "observation_unit": "Social service offered by an equipment",
    "service_records": len(service_locations),
    "distinct_equipment": int(selected_services["equipment_id"].nunique()),
    "service_type_counts": category_counts.to_dict(),
    "point_features": len(service_points),
    "geopackage_available": len(service_points) > 0,
    "geopackage_layer": "service_locations",
    "records_without_valid_points": len(service_locations) - len(service_points),
    "coordinate_reference_system": "EPSG:4326",
    "coordinate_quality": "Availability does not certify positional accuracy.",
}
write_json(dataset_metadata, PROCESSED / "carta_social_dataset_metadata.json")


## Spatial inspection

The map provides an initial diagnostic rather than positional validation.
Inspect unexpected locations and compare a sample against the source
addresses. Different services at the same equipment overlap.

Unresolved records remain in the CSV. Their distribution and any geographic
exclusions should be reported alongside subsequent analysis.


In [ ]:
if len(service_points) > 0:
    service_points.plot(figsize=(8, 9), markersize=3, alpha=0.5)
    plt.title("Carta Social: social-service locations")
    plt.xlabel("Longitude (degrees)")
    plt.ylabel("Latitude (degrees)")
    plt.show()


## Output interpretation

The CSV retains every record in the configured selection, while the
GeoPackage includes only records with valid point geometry. Several
service records may share a facility location. The metadata JSON records
the actual category coverage and the number of records omitted from the map.

Consult the [dataset catalog](../docs/datasets.md) for file definitions,
the [methodology](../docs/methodology.md) for processing assumptions, and
the [methodology](../docs/methodology.md) for observed results.
